# Stage 1: MacBERT 多任務基準模型

訓練 MacBERT multi-task 5-fold baseline，產生 Stage 2 需要的 OOF/test 機率檔與第一版 submission。

預設 Drive root：`/content/drive/MyDrive/VeriPromiseESG2026`。


## 0. Colab 執行環境與路徑檢查

請先執行下一個 setup cell。它會安裝套件、掛載 Google Drive、檢查 GPU、建立輸出資料夾，並驗證必要輸入檔是否存在。


In [ ]:
# Colab 啟動區塊：安裝套件、掛載 Drive、檢查 GPU、驗證必要輸入檔。
!pip -q install transformers accelerate scikit-learn pandas numpy tqdm openpyxl

from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

BASE_DIR = "/content/drive/MyDrive/VeriPromiseESG2026"
DATA_DIR = f"{BASE_DIR}/data"
OUTPUT_ROOT = f"{BASE_DIR}/outputs"
STAGE_NAME = "stage1_macbert_multitask_baseline"
STAGE_DISPLAY_NAME = "Stage 1 - MacBERT 多任務基準模型"
OUTPUT_DIR = f"{OUTPUT_ROOT}/{STAGE_NAME}"
OUT_DIR = OUTPUT_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)

REQUIRED_RELATIVE_PATHS = [
    "data/vpesg4k_train_1000.json",
    "data/vpesg4k_valdata_1000.json",
    "data/vpesg4k_testdata_2000.json",
]

def stage_log(label, value):
    print(f"[{STAGE_NAME}] {label}: {value}")

def require_files(relative_paths):
    missing = []
    for rel in relative_paths:
        path = f"{BASE_DIR}/{rel}"
        if not os.path.exists(path):
            missing.append(path)
    if missing:
        msg = "缺少必要輸入檔。請先執行前一個 Stage，或將資料放到 Google Drive：\n" + "\n".join(missing)
        raise FileNotFoundError(msg)

require_files(REQUIRED_RELATIVE_PATHS)
stage_log("Stage", STAGE_DISPLAY_NAME)
stage_log("BASE_DIR", BASE_DIR)
stage_log("DATA_DIR", DATA_DIR)
stage_log("OUTPUT_DIR", OUTPUT_DIR)

# 完整訓練建議使用 Colab A100 或同級 GPU。
try:
    gpu_names = !nvidia-smi --query-gpu=name --format=csv,noheader
    gpu_names = list(gpu_names)
    stage_log("GPU", gpu_names)
    if not any("A100" in str(name) for name in gpu_names):
        stage_log("WARNING", "建議使用 A100；非 A100 runtime 可能需要更長時間。")
except Exception as exc:
    stage_log("WARNING", f"無法取得 GPU 資訊，請確認 Colab runtime 已啟用 GPU。{exc}")


## 輸入輸出契約

Stage 顯示名稱：`Stage 1 - MacBERT 多任務基準模型`

Google Drive 輸出資料夾：

```text
/content/drive/MyDrive/VeriPromiseESG2026/outputs/stage1_macbert_multitask_baseline/
```

必要輸入檔：

- `data/vpesg4k_train_1000.json`
- `data/vpesg4k_valdata_1000.json`
- `data/vpesg4k_testdata_2000.json`

主要輸出檔：

- `stage1_oof_val_test_probs.pkl`
- `stage1_val1000_predictions.csv`
- `stage1_metrics.json`
- `stage1_baseline_test2000_submission.csv`

若必要輸入不存在，第一個 setup cell 會直接列出缺少的路徑並停止。


## Stage 主要流程

本節訓練 baseline 模型，並以固定檔名輸出後續 Stage 需要的 probability artifact。


## 0. 安裝套件與掛載 Google Drive

## 1. 基本設定：請確認路徑

In [ ]:

import os
import json
import math
import pickle
import random
import re
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, classification_report

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
)

# ====== 你只需要優先確認這裡 ======
BASE_DIR = "/content/drive/MyDrive/VeriPromiseESG2026"
DATA_DIR = f"{BASE_DIR}/data"

TRAIN_PATH = f"{DATA_DIR}/vpesg4k_train_1000.json"
VAL_PATH   = f"{DATA_DIR}/vpesg4k_valdata_1000.json"
TEST_PATH  = f"{DATA_DIR}/vpesg4k_testdata_2000.json"

OUT_DIR = f"{BASE_DIR}/outputs/stage1_macbert_multitask_baseline"
os.makedirs(OUT_DIR, exist_ok=True)

# ====== 模型與訓練設定 ======
MODEL_NAME = "hfl/chinese-macbert-base"
SEED = 42
N_SPLITS = 5
MAX_LEN = 384
EPOCHS = 4
BATCH_SIZE = 8
GRAD_ACCUM = 2
LR = 3e-5
WEIGHT_DECAY = 0.01
PATIENCE = 2
NUM_WORKERS = 2

USE_CLASS_WEIGHTS = True
USE_FOCAL_FOR_T4 = True
FOCAL_GAMMA = 3.0

APPLY_EXACT_MATCH_OVERRIDE = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
print("OUT_DIR:", OUT_DIR)


## 2. 固定隨機種子

In [ ]:

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # 速度優先；若要完全可重現，可改 deterministic=True，但會慢
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)


## 3. 讀取正確資料集

In [ ]:

def load_json_df(path):
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    return pd.DataFrame(obj)

train_df = load_json_df(TRAIN_PATH)
val_df = load_json_df(VAL_PATH)
test_df = load_json_df(TEST_PATH)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

display(train_df.head(2))
display(val_df.head(2))
display(test_df.head(2))


## 4. 標籤與欄位設定

In [ ]:

TASKS = [
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality",
]

LABELS = {
    "promise_status": ["No", "Yes"],
    "verification_timeline": ["N/A", "already", "within_2_years", "between_2_and_5_years", "more_than_5_years"],
    "evidence_status": ["N/A", "No", "Yes"],
    "evidence_quality": ["N/A", "Clear", "Not Clear", "Misleading"],
}

LABEL2ID = {t: {lab: i for i, lab in enumerate(labs)} for t, labs in LABELS.items()}
ID2LABEL = {t: {i: lab for i, lab in enumerate(labs)} for t, labs in LABELS.items()}

TEXT_COL = "data"
ID_COL = "id"

def normalize_label_df(df):
    df = df.copy()
    for t in TASKS:
        if t in df.columns:
            df[t] = df[t].fillna("N/A").astype(str)
            df[t] = df[t].replace({"nan": "N/A", "": "N/A"})
    return df

train_df = normalize_label_df(train_df)
val_df = normalize_label_df(val_df)

for t in TASKS:
    print("\n", t)
    print("train:", train_df[t].value_counts(dropna=False).to_dict())
    print("val:  ", val_df[t].value_counts(dropna=False).to_dict())


## 5. 資料檢視：筆數、ID、重複、階層規則

In [ ]:

def compact_text(s):
    s = str(s)
    s = re.sub(r"\s+", "", s)
    return s

def audit_basic(name, df):
    print(f"\n==== {name} ====")
    print("rows:", len(df))
    print("unique id:", df[ID_COL].nunique() if ID_COL in df.columns else None)
    print("duplicate id rows:", df.duplicated(ID_COL).sum() if ID_COL in df.columns else None)
    print("unique raw data:", df[TEXT_COL].nunique())
    print("duplicate raw data rows:", df.duplicated(TEXT_COL).sum())
    print("id min/max:", df[ID_COL].min(), df[ID_COL].max())

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    audit_basic(name, df)

# cross exact duplicates by compact data
for df in [train_df, val_df, test_df]:
    df["_compact_data"] = df[TEXT_COL].map(compact_text)

def overlap_report(df1, df2, name1, name2):
    id_overlap = set(df1[ID_COL]).intersection(set(df2[ID_COL]))
    data_overlap = set(df1["_compact_data"]).intersection(set(df2["_compact_data"]))
    print(f"\n{name1} vs {name2}")
    print("id overlap:", len(id_overlap))
    print("compact data overlap:", len(data_overlap))
    if data_overlap:
        rows = []
        for text_key in sorted(data_overlap):
            a = df1[df1["_compact_data"] == text_key].head(1)
            b = df2[df2["_compact_data"] == text_key].head(1)
            rows.append({
                f"{name1}_id": a[ID_COL].iloc[0],
                f"{name1}_company": a["company"].iloc[0] if "company" in a.columns else "",
                f"{name1}_page": a["page_number"].iloc[0] if "page_number" in a.columns else "",
                f"{name2}_id": b[ID_COL].iloc[0],
                f"{name2}_company": b["company"].iloc[0] if "company" in b.columns else "",
                f"{name2}_page": b["page_number"].iloc[0] if "page_number" in b.columns else "",
                "data": a[TEXT_COL].iloc[0],
            })
        return pd.DataFrame(rows)
    return pd.DataFrame()

dup_train_val = overlap_report(train_df, val_df, "train", "val")
dup_train_test = overlap_report(train_df, test_df, "train", "test")
dup_val_test = overlap_report(val_df, test_df, "val", "test")

display(dup_train_val)
display(dup_train_test)
display(dup_val_test)

dup_train_test.to_csv(f"{OUT_DIR}/audit_train_test_exact_duplicates.csv", index=False, encoding="utf-8-sig")


In [ ]:

def check_hierarchy_violations(df, name):
    violations = []
    for i, row in df.iterrows():
        rid = row[ID_COL]
        p = row["promise_status"]
        tl = row["verification_timeline"]
        es = row["evidence_status"]
        eq = row["evidence_quality"]

        if p == "No":
            if not (tl == "N/A" and es == "N/A" and eq == "N/A"):
                violations.append((rid, "promise_No_downstream_not_NA", p, tl, es, eq))
        if es in ["No", "N/A"]:
            if eq != "N/A":
                violations.append((rid, "evidence_not_Yes_quality_not_NA", p, tl, es, eq))
    print(name, "violations:", len(violations))
    return pd.DataFrame(violations, columns=["id", "type", "promise_status", "verification_timeline", "evidence_status", "evidence_quality"])

viol_train = check_hierarchy_violations(train_df, "train")
viol_val = check_hierarchy_violations(val_df, "val")

display(viol_train.head())
display(viol_val.head())

viol_train.to_csv(f"{OUT_DIR}/audit_train_hierarchy_violations.csv", index=False, encoding="utf-8-sig")
viol_val.to_csv(f"{OUT_DIR}/audit_val_hierarchy_violations.csv", index=False, encoding="utf-8-sig")


## 6. 評分器：競賽官方 weighted score + 診斷用 macro score

In [ ]:

def apply_hierarchy_to_pred_df(pred_df):
    pred_df = pred_df.copy()
    no_mask = pred_df["promise_status"] == "No"
    pred_df.loc[no_mask, "verification_timeline"] = "N/A"
    pred_df.loc[no_mask, "evidence_status"] = "N/A"
    pred_df.loc[no_mask, "evidence_quality"] = "N/A"

    not_evidence_yes = pred_df["evidence_status"] != "Yes"
    pred_df.loc[not_evidence_yes, "evidence_quality"] = "N/A"
    return pred_df

def f1_yes(y_true, y_pred):
    """
    官方 T1/T3 都是算 Yes 類別的 F1。
    但 T3 evidence_status 原始標籤有 N/A / No / Yes 三類，
    不能直接用 sklearn average="binary"。
    這裡先轉成「是否為 Yes」的二元標籤再算 F1。
    """
    y_true_bin = (pd.Series(y_true).astype(str).values == "Yes").astype(int)
    y_pred_bin = (pd.Series(y_pred).astype(str).values == "Yes").astype(int)
    return f1_score(
        y_true_bin,
        y_pred_bin,
        pos_label=1,
        average="binary",
        zero_division=0,
    )

def macro_f1_task(y_true, y_pred, task):
    return f1_score(
        y_true,
        y_pred,
        labels=LABELS[task],
        average="macro",
        zero_division=0,
    )

def calc_metrics(y_true_df, pred_df, prefix=""):
    pred_df = apply_hierarchy_to_pred_df(pred_df)
    out = {}

    # 官方風格：T1/T3 用 Yes F1，T2/T4 用 macro F1
    out["promise_status_f1_yes"] = f1_yes(y_true_df["promise_status"], pred_df["promise_status"])
    out["verification_timeline_macro_f1"] = macro_f1_task(y_true_df["verification_timeline"], pred_df["verification_timeline"], "verification_timeline")
    out["evidence_status_f1_yes"] = f1_yes(y_true_df["evidence_status"], pred_df["evidence_status"])
    out["evidence_quality_macro_f1"] = macro_f1_task(y_true_df["evidence_quality"], pred_df["evidence_quality"], "evidence_quality")

    out["weighted_score"] = (
        0.20 * out["promise_status_f1_yes"]
        + 0.15 * out["verification_timeline_macro_f1"]
        + 0.30 * out["evidence_status_f1_yes"]
        + 0.35 * out["evidence_quality_macro_f1"]
    )

    # 診斷用：四任務 macro，不作為主分數
    for t in TASKS:
        out[f"{t}_macro_f1_diag"] = macro_f1_task(y_true_df[t], pred_df[t], t)

    if prefix:
        out = {f"{prefix}{k}": v for k, v in out.items()}
    return out

def print_metrics(metrics, title="metrics"):
    print("\n" + "="*80)
    print(title)
    print("="*80)
    for k, v in metrics.items():
        print(f"{k:36s}: {v:.6f}" if isinstance(v, float) else f"{k}: {v}")


## 7. Dataset 與模型：MacBERT + [CLS] + mean pooling + 四任務頭

In [ ]:

class ESGDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=384, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row[TEXT_COL])

        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors=None,
        )

        item = {
            "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
        }

        if "token_type_ids" in enc:
            item["token_type_ids"] = torch.tensor(enc["token_type_ids"], dtype=torch.long)

        if self.has_labels:
            labels = {}
            for t in TASKS:
                labels[t] = torch.tensor(LABEL2ID[t][row[t]], dtype=torch.long)
            item["labels"] = labels

        return item

class MultiTaskClassifier(nn.Module):
    def __init__(self, model_name, label_dims, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        pooled_dim = hidden * 2  # CLS + mean pooling

        self.dropouts = nn.ModuleDict({
            t: nn.Dropout(dropout) for t in TASKS
        })
        self.classifiers = nn.ModuleDict({
            t: nn.Linear(pooled_dim, label_dims[t]) for t in TASKS
        })

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        }
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids

        out = self.encoder(**kwargs)
        last_hidden = out.last_hidden_state

        cls_vec = last_hidden[:, 0, :]

        mask = attention_mask.unsqueeze(-1).float()
        mean_vec = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)

        pooled = torch.cat([cls_vec, mean_vec], dim=-1)

        logits = {}
        for t in TASKS:
            logits[t] = self.classifiers[t](self.dropouts[t](pooled))
        return logits


## 8. Loss：class weight + T4 focal loss

In [ ]:

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits, target):
        ce = nn.functional.cross_entropy(logits, target, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()

def make_class_weights(fold_train_df):
    weights = {}
    for t in TASKS:
        counts = fold_train_df[t].value_counts().to_dict()
        arr = []
        total = len(fold_train_df)
        n_classes = len(LABELS[t])
        for lab in LABELS[t]:
            c = counts.get(lab, 0)
            # 避免某 fold 類別完全缺失造成無限大
            c = max(c, 1)
            arr.append(total / (n_classes * c))
        w = torch.tensor(arr, dtype=torch.float)
        # 避免極端少數類權重過大，先做上限保護
        w = torch.clamp(w, max=10.0)
        weights[t] = w.to(DEVICE)
    return weights

def make_losses(fold_train_df):
    class_weights = make_class_weights(fold_train_df) if USE_CLASS_WEIGHTS else {t: None for t in TASKS}
    losses = {}
    for t in TASKS:
        if t == "evidence_quality" and USE_FOCAL_FOR_T4:
            losses[t] = FocalLoss(gamma=FOCAL_GAMMA, weight=class_weights[t])
        else:
            losses[t] = nn.CrossEntropyLoss(weight=class_weights[t])
    return losses


## 9. 預測、轉標籤、錯誤分析工具

In [ ]:

def batch_to_device(batch):
    out = {}
    for k, v in batch.items():
        if k == "labels":
            out[k] = {t: y.to(DEVICE) for t, y in v.items()}
        else:
            out[k] = v.to(DEVICE)
    return out

@torch.no_grad()
def predict_probs(model, df, tokenizer, batch_size=16):
    model.eval()
    ds = ESGDataset(df, tokenizer, max_len=MAX_LEN, has_labels=False)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)

    all_probs = {t: [] for t in TASKS}
    for batch in tqdm(dl, desc="predict", leave=False):
        batch = batch_to_device(batch)
        logits = model(**batch)
        for t in TASKS:
            probs = torch.softmax(logits[t], dim=-1).detach().cpu().numpy()
            all_probs[t].append(probs)

    all_probs = {t: np.concatenate(all_probs[t], axis=0) for t in TASKS}
    return all_probs

def probs_to_pred_df(df, probs, include_id=True):
    pred = pd.DataFrame()
    if include_id:
        pred[ID_COL] = df[ID_COL].values
    for t in TASKS:
        ids = probs[t].argmax(axis=1)
        pred[t] = [ID2LABEL[t][int(i)] for i in ids]
    pred = apply_hierarchy_to_pred_df(pred)
    return pred

def save_confusion_matrices(y_true_df, pred_df, out_xlsx):
    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        for t in TASKS:
            labs = LABELS[t]
            cm = confusion_matrix(y_true_df[t], pred_df[t], labels=labs)
            cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in labs], columns=[f"pred_{x}" for x in labs])
            cm_df.to_excel(writer, sheet_name=t[:31])
    print("saved:", out_xlsx)

def make_error_analysis(y_true_df, pred_df):
    rows = []
    for i in range(len(y_true_df)):
        r = {
            ID_COL: y_true_df.iloc[i][ID_COL],
            TEXT_COL: y_true_df.iloc[i][TEXT_COL],
            "company": y_true_df.iloc[i].get("company", ""),
            "page_number": y_true_df.iloc[i].get("page_number", ""),
        }
        any_err = False
        for t in TASKS:
            yt = y_true_df.iloc[i][t]
            yp = pred_df.iloc[i][t]
            r[f"true_{t}"] = yt
            r[f"pred_{t}"] = yp
            r[f"err_{t}"] = bool(yt != yp)
            any_err = any_err or (yt != yp)
        r["any_error"] = any_err
        rows.append(r)
    return pd.DataFrame(rows)


## 10. 單 fold 訓練函式

In [ ]:

def evaluate_model_on_df(model, df, tokenizer, title="eval"):
    probs = predict_probs(model, df, tokenizer, batch_size=16)
    pred = probs_to_pred_df(df, probs, include_id=True)
    metrics = calc_metrics(df, pred)
    print_metrics(metrics, title)
    return metrics, pred, probs

def train_one_fold(fold, fold_train_df, fold_valid_df, tokenizer):
    print(f"\n========== Fold {fold} ==========")
    print("train:", fold_train_df.shape, "valid:", fold_valid_df.shape)

    model = MultiTaskClassifier(
        MODEL_NAME,
        label_dims={t: len(LABELS[t]) for t in TASKS},
        dropout=0.2,
    ).to(DEVICE)

    train_ds = ESGDataset(fold_train_df, tokenizer, max_len=MAX_LEN, has_labels=True)
    valid_ds = ESGDataset(fold_valid_df, tokenizer, max_len=MAX_LEN, has_labels=False)

    train_dl = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    total_steps = math.ceil(len(train_dl) / GRAD_ACCUM) * EPOCHS
    warmup_steps = int(total_steps * 0.1)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    losses = make_losses(fold_train_df)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

    best_score = -1
    best_path = f"{OUT_DIR}/fold{fold}_best.pt"
    bad_epochs = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        optimizer.zero_grad(set_to_none=True)

        pbar = tqdm(train_dl, desc=f"fold{fold} epoch{epoch}", leave=False)
        for step, batch in enumerate(pbar, start=1):
            batch = batch_to_device(batch)
            labels = batch.pop("labels")

            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                logits = model(**batch)
                loss = 0
                for t in TASKS:
                    loss = loss + losses[t](logits[t], labels[t])
                loss = loss / len(TASKS)
                loss = loss / GRAD_ACCUM

            scaler.scale(loss).backward()
            total_loss += loss.item() * GRAD_ACCUM

            if step % GRAD_ACCUM == 0 or step == len(train_dl):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            pbar.set_postfix(loss=total_loss / step)

        # fold valid 評分
        valid_probs = predict_probs(model, fold_valid_df, tokenizer, batch_size=16)
        valid_pred = probs_to_pred_df(fold_valid_df, valid_probs, include_id=True)
        metrics = calc_metrics(fold_valid_df, valid_pred)
        score = metrics["weighted_score"]

        print_metrics(metrics, title=f"fold {fold} epoch {epoch} valid")

        if score > best_score:
            best_score = score
            bad_epochs = 0
            torch.save(model.state_dict(), best_path)
            print(f"saved best fold{fold}: {best_score:.6f}")
        else:
            bad_epochs += 1
            print("bad_epochs:", bad_epochs)
            if bad_epochs >= PATIENCE:
                print("early stopping")
                break

    # 載入最佳權重
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    return model, best_score, best_path


## 11. 5-fold 訓練：只用 train1000 訓練，val1000 留作固定驗證

In [ ]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# StratifiedKFold 用較穩的複合鍵，避免 Misleading 只有 1 筆造成切分失敗
strat_key = train_df["promise_status"].astype(str) + "|" + train_df["evidence_status"].astype(str)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# 儲存 OOF / val / test 機率
oof_probs = {t: np.zeros((len(train_df), len(LABELS[t])), dtype=np.float32) for t in TASKS}
val_probs_sum = {t: np.zeros((len(val_df), len(LABELS[t])), dtype=np.float32) for t in TASKS}
test_probs_sum = {t: np.zeros((len(test_df), len(LABELS[t])), dtype=np.float32) for t in TASKS}

fold_scores = []
fold_paths = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, strat_key), start=1):
    fold_train_df = train_df.iloc[tr_idx].reset_index(drop=True)
    fold_valid_df = train_df.iloc[va_idx].reset_index(drop=True)

    model, best_score, best_path = train_one_fold(fold, fold_train_df, fold_valid_df, tokenizer)
    fold_scores.append(best_score)
    fold_paths.append(best_path)

    # OOF probs
    fold_valid_probs = predict_probs(model, fold_valid_df, tokenizer, batch_size=16)
    for t in TASKS:
        oof_probs[t][va_idx] = fold_valid_probs[t]

    # val1000 probs
    val_probs = predict_probs(model, val_df, tokenizer, batch_size=16)
    for t in TASKS:
        val_probs_sum[t] += val_probs[t] / N_SPLITS

    # test2000 probs
    test_probs = predict_probs(model, test_df, tokenizer, batch_size=16)
    for t in TASKS:
        test_probs_sum[t] += test_probs[t] / N_SPLITS

    # 釋放 GPU
    del model
    torch.cuda.empty_cache()

print("fold_scores:", fold_scores)
print("mean fold score:", np.mean(fold_scores))

# 儲存 raw probs
prob_obj = {
    "model_name": MODEL_NAME,
    "tasks": TASKS,
    "labels": LABELS,
    "train_path": TRAIN_PATH,
    "val_path": VAL_PATH,
    "test_path": TEST_PATH,
    "fold_paths": fold_paths,
    "fold_scores": fold_scores,
    "oof_probs": oof_probs,
    "val_probs": val_probs_sum,
    "test_probs": test_probs_sum,
    "train_ids": train_df[ID_COL].tolist(),
    "val_ids": val_df[ID_COL].tolist(),
    "test_ids": test_df[ID_COL].tolist(),
}
with open(f"{OUT_DIR}/stage1_oof_val_test_probs.pkl", "wb") as f:
    pickle.dump(prob_obj, f)

print("saved probs:", f"{OUT_DIR}/stage1_oof_val_test_probs.pkl")


## 12. OOF / val1000 評分

In [ ]:

# train OOF
oof_pred = probs_to_pred_df(train_df, oof_probs, include_id=True)
oof_metrics = calc_metrics(train_df, oof_pred)
print_metrics(oof_metrics, "TRAIN1000 OOF metrics")

oof_pred_full = pd.concat(
    [train_df[[ID_COL, TEXT_COL, "company", "page_number"]].reset_index(drop=True),
     oof_pred[TASKS].add_prefix("pred_").reset_index(drop=True)],
    axis=1,
)
for t in TASKS:
    oof_pred_full[f"true_{t}"] = train_df[t].values

oof_pred_full.to_csv(f"{OUT_DIR}/stage1_train1000_oof_predictions.csv", index=False, encoding="utf-8-sig")
save_confusion_matrices(train_df, oof_pred, f"{OUT_DIR}/stage1_train1000_oof_confusion.xlsx")

# val1000
val_pred = probs_to_pred_df(val_df, val_probs_sum, include_id=True)
val_metrics = calc_metrics(val_df, val_pred)
print_metrics(val_metrics, "VAL1000 metrics")

val_pred_full = pd.concat(
    [val_df[[ID_COL, TEXT_COL, "company", "page_number"]].reset_index(drop=True),
     val_pred[TASKS].add_prefix("pred_").reset_index(drop=True)],
    axis=1,
)
for t in TASKS:
    val_pred_full[f"true_{t}"] = val_df[t].values

val_pred_full.to_csv(f"{OUT_DIR}/stage1_val1000_predictions.csv", index=False, encoding="utf-8-sig")
save_confusion_matrices(val_df, val_pred, f"{OUT_DIR}/stage1_val1000_confusion.xlsx")

err_df = make_error_analysis(val_df, val_pred)
err_df.to_csv(f"{OUT_DIR}/stage1_val1000_error_analysis.csv", index=False, encoding="utf-8-sig")
err_df.to_excel(f"{OUT_DIR}/stage1_val1000_error_analysis.xlsx", index=False)

metrics_obj = {
    "oof_metrics": oof_metrics,
    "val1000_metrics": val_metrics,
    "fold_scores": fold_scores,
}
with open(f"{OUT_DIR}/stage1_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_obj, f, ensure_ascii=False, indent=2)

print("saved all evaluation outputs to:", OUT_DIR)
display(err_df["any_error"].value_counts())
display(err_df.head())


## 13. 產生 test2000 submission

In [ ]:

test_pred = probs_to_pred_df(test_df, test_probs_sum, include_id=True)

# exact match override：若 test 段落與 train 完全相同，直接用 train label 覆蓋
if APPLY_EXACT_MATCH_OVERRIDE:
    train_map = {}
    for _, row in train_df.iterrows():
        train_map[row["_compact_data"]] = {t: row[t] for t in TASKS}

    override_count = 0
    for i, row in test_df.iterrows():
        key = row["_compact_data"]
        if key in train_map:
            for t in TASKS:
                test_pred.loc[i, t] = train_map[key][t]
            override_count += 1

    test_pred = apply_hierarchy_to_pred_df(test_pred)
    print("exact-match override count:", override_count)

# 儲存 submission
submission_cols = [ID_COL] + TASKS
sub = test_pred[submission_cols].copy()

sub_path = f"{OUT_DIR}/stage1_baseline_test2000_submission.csv"
sub.to_csv(sub_path, index=False, encoding="utf-8-sig")
print("saved:", sub_path)

# 分布檢查
for t in TASKS:
    print("\n", t)
    print(sub[t].value_counts().to_dict())

display(sub.head())


## 14. 產生報告摘要

In [ ]:

report_lines = []
report_lines.append("# stage1 val1000 MacBERT multitask clean pipeline report")
report_lines.append("")
report_lines.append(f"- model_name: {MODEL_NAME}")
report_lines.append(f"- train_path: {TRAIN_PATH}")
report_lines.append(f"- val_path: {VAL_PATH}")
report_lines.append(f"- test_path: {TEST_PATH}")
report_lines.append(f"- out_dir: {OUT_DIR}")
report_lines.append("")
report_lines.append("## Fold scores")
for i, s in enumerate(fold_scores, start=1):
    report_lines.append(f"- fold{i}: {s:.6f}")
report_lines.append(f"- mean: {np.mean(fold_scores):.6f}")
report_lines.append("")
report_lines.append("## Train1000 OOF metrics")
for k, v in oof_metrics.items():
    report_lines.append(f"- {k}: {v:.6f}")
report_lines.append("")
report_lines.append("## Val1000 metrics")
for k, v in val_metrics.items():
    report_lines.append(f"- {k}: {v:.6f}")
report_lines.append("")
report_lines.append("## Test prediction distribution")
for t in TASKS:
    report_lines.append(f"### {t}")
    for lab, cnt in sub[t].value_counts().to_dict().items():
        report_lines.append(f"- {lab}: {cnt}")

report_md = "\n".join(report_lines)
report_path = f"{OUT_DIR}/stage1_report.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_md)

print(report_md[:3000])
print("saved:", report_path)



## 15. 下一步

跑完這份之後，請先看：

1. `stage1_metrics.json`
2. `stage1_val1000_error_analysis.xlsx`
3. `stage1_val1000_confusion.xlsx`
4. `stage1_baseline_test2000_submission.csv`

如果 val1000 分數穩定，下一版建議做：

- `taskwise_ensemble`：加入 CKIP / MacBERT 多 stem 融合
- `t2_t4_specialists`：T2 timeline specialist、T4 quality specialist
- `3view_TTA`：head / middle / tail 文字視角
